In [1]:
import pandas as pd
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from konlpy.tag import Mecab
from gensim.models import Word2Vec, KeyedVectors
from tqdm.auto import tqdm
import math

# CSV 파일 로드
data = pd.read_csv('./data/ChatbotData.csv')
data = data.drop_duplicates(subset='Q').reset_index(drop=True)
data = data.drop_duplicates(subset='A').reset_index(drop=True)

print(f"전체 데이터 수: {len(data)}")

ModuleNotFoundError: No module named 'konlpy'

In [2]:
questions = []
for sentence in data['Q'].tolist():
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = sentence.strip()
    questions.append(sentence)

answers = []
for sentence in data['A'].tolist():
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = sentence.strip()
    answers.append(sentence)

In [5]:
# 1. 사전 훈련된 Word2Vec 모델 로드
wv = KeyedVectors.load_word2vec_format('./w2v/ko_c.bin', binary=True)

# 형태소 분석기로 사용
mecab = Mecab(dicpath="/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic")

# 2. Lexical Substitution 함수 정의
def lexical_sub(sentence_tokens, wv_model, target_pos=['NNG', 'VV', 'VA', 'MAG'], is_print=False):

    if wv_model is None:
        return sentence_tokens

    pos_tagged = mecab.pos(" ".join(sentence_tokens))
    if is_print:
        print(pos_tagged)

    valid_tokens = [word for word, pos in pos_tagged if any(pos.startswith(p) for p in target_pos) and word in wv_model.key_to_index]
    if not valid_tokens:
        return sentence_tokens

    selected_tok = random.choice(valid_tokens)

    try:
        similar_words = wv_model.most_similar(selected_tok, topn=20)

        similar_word = None # 최종 선택될 단어를 저장할 변수
        for word, _ in similar_words:
            new_word_pos_list = mecab.pos(word)

            if new_word_pos_list and any(tagged_part[1].startswith(p) for tagged_part in new_word_pos_list for p in target_pos):
                similar_word = word
                break

        # for문을 모두 돌았는데도 적절한 단어를 못 찾았을 경우, 그냥 가장 비슷한 단어로 대체
        if similar_word is None:
            similar_word = similar_words[0][0]

    except KeyError:
        return sentence_tokens

    new_sentence_tokens = [similar_word if tok == selected_tok else tok for tok in sentence_tokens]

    return new_sentence_tokens

# 3. 데이터 증강 실행
# aug_que_corpus = [lexical_sub(q, wv) for q in tqdm(questions, desc="Augmenting Questions")]
# aug_ans_corpus = [lexical_sub(a, wv) for a in tqdm(answers, desc="Augmenting Answers")]

aug_que_corpus = [" ".join(lexical_sub(mecab.morphs(q), wv)) for q in tqdm(questions, desc="Augmenting Questions")]
aug_ans_corpus = [" ".join(lexical_sub(mecab.morphs(a), wv)) for a in tqdm(answers, desc="Augmenting Answers")]

# 4. 원본 데이터와 증강 데이터 결합
final_que_corpus = questions + aug_que_corpus + questions
final_ans_corpus = answers + answers + aug_ans_corpus

print(f"원본 데이터 크기: {len(questions)}")
print(f"증강 후 데이터 크기: {len(final_que_corpus)}")

Augmenting Questions:   0%|          | 0/7731 [00:00<?, ?it/s]

Augmenting Answers:   0%|          | 0/7731 [00:00<?, ?it/s]

원본 데이터 크기: 7731
증강 후 데이터 크기: 23193


In [6]:
import sentencepiece as spm

with open('all.txt', 'w', encoding='utf8') as f:
    f.write('\n'.join(final_que_corpus))
    f.write('\n'.join(final_ans_corpus))

In [14]:
corpus = "all.txt"
prefix = "chatbot"

vocab_size = 6000
MAX_LEN = 40

spm.SentencePieceTrainer.train(
    f"--input={corpus} --model_prefix={prefix} --vocab_size={vocab_size}" +
    f" --model_type=bpe" +
    f" --max_sentence_length={MAX_LEN}" # 문장 최대 길이
    " --pad_id=0 --pad_piece=[PAD]" # pad (0)
    " --unk_id=1 --unk_piece=[UNK]" # unknown (1)
    " --bos_id=2 --bos_piece=[BOS]" # begin of sequence (2)
    " --eos_id=3 --eos_piece=[EOS]" # end of sequence (3)
    # " --user_defined_symbols=[SEP],[CLS],[MASK]"  # 사용자 정의 토큰
)

sentencepiece_trainer.cc(177) LOG(INFO) Running command: --input=all.txt --model_prefix=chatbot --vocab_size=6000 --model_type=bpe --max_sentence_length=40 --pad_id=0 --pad_piece=[PAD] --unk_id=1 --unk_piece=[UNK] --bos_id=2 --bos_piece=[BOS] --eos_id=3 --eos_piece=[EOS]
sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: all.txt
  input_format: 
  model_prefix: chatbot
  model_type: BPE
  vocab_size: 6000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 40
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  hard_vocab_limit: 1

In [15]:
vocab_file = "chatbot.model"
vocab = spm.SentencePieceProcessor()
vocab.load(vocab_file)

True

In [16]:
line = "안녕하세요 만나서 반갑습니다"
pieces = vocab.encode_as_pieces(line)
ids = vocab.encode_as_ids(line)


print(line)
print(pieces)
print(ids)

안녕하세요 만나서 반갑습니다
['▁안녕', '하세요', '▁만나서', '▁반갑', '습니다']
[965, 181, 4098, 4672, 151]


In [19]:
START_TOKEN = [2]
END_TOKEN = [3]

# 토큰화 / 정수 인코딩 / 시작 토큰과 종료 토큰 추가 / 패딩
def tokenize_and_filter(inputs, outputs):
  tokenized_inputs, tokenized_outputs = [], []

  for (sentence1, sentence2) in zip(inputs, outputs):
    # encode(토큰화 + 정수 인코딩), 시작 토큰과 종료 토큰 추가
    zeros1 = np.zeros(MAX_LEN, dtype=int)
    zeros2 = np.zeros(MAX_LEN, dtype=int)
    sentence1 = START_TOKEN + vocab.encode_as_ids(sentence1) + END_TOKEN
    zeros1[:len(sentence1)] = sentence1[:MAX_LEN]

    sentence2 = START_TOKEN + vocab.encode_as_ids(sentence2) + END_TOKEN
    zeros2[:len(sentence2)] = sentence2[:MAX_LEN]

    tokenized_inputs.append(zeros1)
    tokenized_outputs.append(zeros2)
  return tokenized_inputs, tokenized_outputs

questions_encode, answers_encode = tokenize_and_filter(final_que_corpus, final_ans_corpus)
print(questions_encode[0])
print(answers_encode[0])

[   2 2743 4975 2022   74    3    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]
[   2 3548  136   16   22    4    3    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]


In [21]:
from torch.utils.data import Dataset, DataLoader

class SequenceDataset(Dataset):
    def __init__(self, questions, answers):
        questions = np.array(questions)
        answers = np.array(answers)
        self.inputs = questions
        self.dec_inputs = answers[:,:-1]
        self.outputs = answers[:,1:]
        self.length = len(questions)

    def __getitem__(self,idx):
        return (self.inputs[idx], self.dec_inputs[idx], self.outputs[idx])

    def __len__(self):
        return self.length

BATCH_SIZE = 128
dataset = SequenceDataset(questions_encode, answers_encode)
dataloader = DataLoader(dataset, shuffle=True, batch_size=BATCH_SIZE)

In [66]:
from torch.nn import Transformer
from torch import nn
import torch
import math

class TFModel(nn.Module):
    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):
        super(TFModel, self).__init__()
        self.transformer = Transformer(ninp, nhead, dim_feedforward=nhid, num_encoder_layers=nlayers, num_decoder_layers=nlayers,dropout=dropout)
        self.pos_encoder = PositionalEncoding(ninp, dropout)
        self.encoder = nn.Embedding(ntoken, ninp)

        self.pos_encoder_d = PositionalEncoding(ninp, dropout)
        self.encoder_d = nn.Embedding(ntoken, ninp)

        self.ninp = ninp
        self.ntoken = ntoken

        self.linear = nn.Linear(ninp, ntoken)
        self.init_weights()

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def init_weights(self):
        initrange = 0.1
        self.encoder.weight.data.uniform_(-initrange, initrange)

    def forward(self, src, tgt, srcmask, tgtmask, srcpadmask, tgtpadmask):
        src = self.encoder(src) * math.sqrt(self.ninp)
        src = self.pos_encoder(src)

        tgt = self.encoder_d(tgt) * math.sqrt(self.ninp)
        tgt = self.pos_encoder_d(tgt)


        output = self.transformer(src.transpose(0,1), tgt.transpose(0,1), srcmask, tgtmask, src_key_padding_mask=srcpadmask, tgt_key_padding_mask=tgtpadmask)
        output = self.linear(output)
        return output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

def gen_attention_mask(x):
    mask = torch.eq(x, 0)
    return mask

In [79]:
# 하이퍼파라미터 설정
N_LAYERS = 2
D_MODEL = 256
N_HEADS = 8
D_FF = D_MODEL * 2
DROPOUT = 0.1
EPOCHS = 30
BATCH_SIZE = 128

In [81]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

lr = 2e-4
model = TFModel(vocab_size, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Using device: cuda


In [82]:
from tqdm.auto import tqdm

epoch = 30

model.train()
for i in range(epoch):
    batchloss = 0.0
    progress = tqdm(dataloader)
    for (inputs, dec_inputs, outputs) in progress:
        optimizer.zero_grad()
        src_mask = model.generate_square_subsequent_mask(MAX_LEN).to(device)
        src_padding_mask = gen_attention_mask(inputs).to(device)
        tgt_mask = model.generate_square_subsequent_mask(MAX_LEN-1).to(device)
        tgt_padding_mask = gen_attention_mask(dec_inputs).to(device)

        result = model(inputs.to(device), dec_inputs.to(device), src_mask, tgt_mask, src_padding_mask,tgt_padding_mask)
        loss = criterion(result.permute(1,2,0), outputs.to(device).long())
        progress.set_description("{:0.3f}".format(loss))
        loss.backward()
        optimizer.step()
        batchloss += loss
    print("epoch:",i+1,"|","loss:",batchloss.cpu().item() / len(dataloader))

  0%|          | 0/182 [00:00<?, ?it/s]

epoch: 1 | loss: 1.7356880942543784


  0%|          | 0/182 [00:00<?, ?it/s]

epoch: 2 | loss: 1.2637725662399124


  0%|          | 0/182 [00:00<?, ?it/s]

...

epoch: 28 | loss: 0.306034905569894


  0%|          | 0/182 [00:00<?, ?it/s]

epoch: 29 | loss: 0.28756822858537945


  0%|          | 0/182 [00:00<?, ?it/s]

epoch: 30 | loss: 0.27102728204412774


In [83]:
def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = sentence.strip()
    return sentence

def evaluate(sentence):
    sentence = preprocess_sentence(sentence)
    input = torch.tensor([START_TOKEN + vocab.encode_as_ids(sentence) + END_TOKEN]).to(device)
    output = torch.tensor([START_TOKEN]).to(device)

    # 디코더의 예측 시작
    model.eval()
    for i in range(MAX_LEN):
        src_mask = model.generate_square_subsequent_mask(input.shape[1]).to(device)
        tgt_mask = model.generate_square_subsequent_mask(output.shape[1]).to(device)

        src_padding_mask = gen_attention_mask(input).to(device)
        tgt_padding_mask = gen_attention_mask(output).to(device)

        predictions = model(input, output, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask).transpose(0,1)
        # 현재(마지막) 시점의 예측 단어를 받아온다.
        predictions = predictions[:, -1:, :]
        predicted_id = torch.LongTensor(torch.argmax(predictions.cpu(), axis=-1))


        # 만약 마지막 시점의 예측 단어가 종료 토큰이라면 예측을 중단
        if torch.equal(predicted_id[0][0], torch.tensor(END_TOKEN[0])):
            break

        # 마지막 시점의 예측 단어를 출력에 연결한다.
        # 이는 for문을 통해서 디코더의 입력으로 사용될 예정이다.
        output = torch.cat([output, predicted_id.to(device)], axis=1)

    return torch.squeeze(output, axis=0).cpu().numpy()

def predict(sentence):
    prediction = evaluate(sentence)
    predicted_sentence = vocab.Decode(list(map(int,[i for i in prediction if i < vocab_size+7])))

    return predicted_sentence

In [84]:
# 예문 리스트
examples = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

# 답변 생성 및 출력
print("--- Translations ---")
for ex in examples:
    print(f"Question: {ex}")
    response = predict(ex)
    print(f"Response: {response}\n")

print("\n--- Hyperparameters ---")
print(f"n_layers: {N_LAYERS}")
print(f"d_model: {D_MODEL}")
print(f"n_heads: {N_HEADS}")
print(f"d_ff: {D_FF}")
print(f"dropout: {DROPOUT}")

print("\n--- Training Parameters ---")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epoch At: {EPOCHS}")

--- Translations ---
Question: 지루하다, 놀러가고 싶어.
Response: 다른 생각을 같이 질투가 않으니까요 .

Question: 오늘 일찍 일어났더니 피곤하다.
Response: 이제 취업이랑 연애였던일 수도 있겠어요 .

Question: 간만에 여자친구랑 데이트 하기로 했어.
Response: 이제라도 틈틈가기도 하죠 .

Question: 집에 있는다는 소리야.
Response: 그것 또한 부담스러워하지 않는다면 그게 최선의 부분이 되겠죠 .


--- Hyperparameters ---
n_layers: 2
d_model: 256
n_heads: 8
d_ff: 512
dropout: 0.1

--- Training Parameters ---
Batch Size: 128
Epoch At: 30


In [85]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    """
    reference: list of tokens (정답 문장)
    candidate: list of tokens (모델이 생성한 문장)
    """
    return sentence_bleu([reference],  # sentence_bleu는 여러 개의 정답을 받을 수 있어 리스트로 감싸줍니다.
                         candidate,
                         weights=weights,
                         smoothing_function=SmoothingFunction().method1)

# 테스트할 문장과 정답(Reference) 설정
test_question = "오늘 정말 최악의 하루였어."
reference_answer_tokens = mecab.morphs("힘들었겠네요. 무슨 일 있었어요?")

# 모델이 생성한 답변 (Candidate)
candidate_answer = predict(test_question)
candidate_answer_tokens = candidate_answer.split()

# BLEU 점수 계산
bleu_score = calculate_bleu(reference_answer_tokens, candidate_answer_tokens)

print(f"질문: {test_question}")
print(f"정답 답변 (Reference): {' '.join(reference_answer_tokens)}")
print(f"모델 생성 답변 (Candidate): {candidate_answer}")
print(f"BLEU Score: {bleu_score:.4f}")

질문: 오늘 정말 최악의 하루였어.
정답 답변 (Reference): 힘들 었 겠 네요 . 무슨 일 있 었 어요 ?
모델 생성 답변 (Candidate): 이제 정말 끝인가봐요 .
BLEU Score: 0.0140


In [87]:
for i in range(5):
    question = final_que_corpus[i]
    reference_answer = final_ans_corpus[i]

    # 모델을 통해 candidate 답변 실시간 생성
    candidate_answer = predict(question)

    # 정답과 생성된 답변을 mecab으로 토큰화
    # BLEU 점수를 계산할 때는 mecab으로 토큰화
    reference_tokens = mecab.morphs(reference_answer)
    candidate_tokens = mecab.morphs(candidate_answer)

    # BLEU 점수 계산
    bleu_score = calculate_bleu(reference_tokens, candidate_tokens) * 100

    # 최종 결과 정리 출력
    print(f"\n[평가 예시 {i+1}]")
    print(f'질문: {question}')
    print(f"정답 답변 (Reference): {reference_answer}")
    print(f"모델 생성 답변 (Candidate): {candidate_answer}")
    print(f"BLEU Score: {bleu_score:.2f}%")
    print("="*30)


[평가 예시 1]
질문: 12시 땡 !
정답 답변 (Reference): 하루가 또 가네요 .
모델 생성 답변 (Candidate): 그리움 을 따지 지 마세요 .
BLEU Score: 4.08%

[평가 예시 2]
질문: 1지망 학교 떨어졌어
정답 답변 (Reference): 위로해 드립니다 .
모델 생성 답변 (Candidate): 위로를 아는 것도 중요합니다 .
BLEU Score: 2.85%

[평가 예시 3]
질문: 3박4일 놀러가고 싶다
정답 답변 (Reference): 여행은 언제나 좋죠 .
모델 생성 답변 (Candidate): 여행 은 항상 좋 은 사람 은 그대 이 에요 .
BLEU Score: 4.74%

[평가 예시 4]
질문: PPL 심하네
정답 답변 (Reference): 눈살이 찌푸려지죠 .
모델 생성 답변 (Candidate): 눈 눈에 대한 눈을 접어두는게 좋겠어요 .
BLEU Score: 1.32%

[평가 예시 5]
질문: SD카드 망가졌어
정답 답변 (Reference): 다시 새로 사는 게 마음 편해요 .
모델 생성 답변 (Candidate): 다시 새로 데이트를 삭제하는게 덜하는게 어떨까요 .
BLEU Score: 4.18%


### Mecab 대신 SPM 사용
- 토크나이저만 바꿨을 뿐인데도 육안으로 봤을 때의 답변, loss가 눈에띄게 좋아졌다.
- 데이터가 적은 상황에서는 subword로 토큰화하는게 효과가 좋아서 그런 것 같다.

### 증강할 때, 특정 품사 내에서 증강
- 기존에는 랜덤한 토큰을 품사 상관없이 가장 유사도가 높은 토큰으로 대체했었음
- 이렇게 되면 문법적으로 완전히 틀린 경우가 많아지게 된다. -> 답변의 문법적인 완성도가 떨어짐
- NNG(일반 명사), VV(동사), VA(형용사), MAG(일반 부사)에 대해서만 증강을 적용함
- 대체해서 새로 생기는 단어도 대체되는 단어와 동일한 품사로 설정
- BLEU 점수의 개선은 미미했지만, 육안상 문법적인 완성도가 좋아짐

### 개선방향
- Greedy Search 대신 Beam Search를 사용해서 생성하는 방법 시도
- dataloader 상에서 증강을 적용해 매 batch마다 새로운 증강이 적용되게 하는 방법 시도
- nn.Transformer를 사용하지 않고 nn.TransformerDecoderLayer, nn.TransformerEncoderLayer를 사용하거나 직접 구현해서
- Attention Map을 보면서 개선 시도